- **3 tools**: `search_internal_docs`, `query_database` (read-only SQLite), `send_email` (mocked — logs instead of sending).
- **Multi-turn agent loop** with `max_turns` and structured error handling.
- **Tool chaining scenario**: look up a customer's last order, then email a status update.
- **Role-based access control**: `agent` vs `admin` roles see/can-call different tools.
- **Deliberate failure cases**: unauthorized tool call, bad arguments, simulated timeout.

Uses OpenAI if `OPENAI_API_KEY` is set. Without a key, a small **scripted
stub LLM** simulates tool-call decisions (keyword-triggered) so the whole
pipeline — registry, RBAC, execution, chaining, error handling — is still
testable end to end.

In [1]:
import os
import json
import sqlite3
import time
import random

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
print("OpenAI key detected:", bool(OPENAI_API_KEY))


OpenAI key detected: False


Mock database

A tiny read-only `orders` table. `query_database` will only ever run
`SELECT` statements against it.

In [2]:
conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE orders (
    id TEXT PRIMARY KEY,
    customer_email TEXT,
    status TEXT,
    total REAL,
    created_at TEXT
)
""")
conn.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?, ?)",
    [
        ("ord_1001", "priya@example.com", "shipped", 49.99, "2026-07-20"),
        ("ord_1002", "priya@example.com", "delivered", 120.00, "2026-07-28"),
        ("ord_1003", "arun@example.com", "pending", 15.50, "2026-08-01"),
    ],
)
conn.commit()
print("Mock DB ready.")


Mock DB ready.


Internal-docs corpus (reused pattern from the RAG capstone)

Keyword search instead of embeddings here, to keep this notebook
dependency-light — swap in the RAG capstone's retriever for a real system.

In [3]:
internal_docs = [
    {"id": "refund_policy#p2", "text": "Damaged or defective items may be refunded within 30 days of delivery."},
    {"id": "shipping_policy#p1", "text": "Standard shipping takes 3-5 business days within the continental US."},
    {"id": "escalation_policy#p1", "text": "Orders pending more than 5 business days should be escalated to the logistics team."},
]

def search_internal_docs(query: str, max_results: int = 3) -> dict:
    query_terms = set(query.lower().split())
    scored = []
    for doc in internal_docs:
        overlap = len(query_terms & set(doc["text"].lower().split()))
        if overlap:
            scored.append((overlap, doc))
    scored.sort(key=lambda x: -x[0])
    top = [d for _, d in scored[:max_results]]
    return {"results": top if top else "No matching internal documents found."}


Tool implementations

In [4]:
ALLOWED_TABLES = {"orders"}

def query_database(sql: str) -> dict:
    sql_clean = sql.strip().lower()
    if not sql_clean.startswith("select"):
        return {"error": "Only SELECT statements are allowed."}
    if not any(t in sql_clean for t in ALLOWED_TABLES):
        return {"error": "Query does not reference an allowed table."}
    try:
        cur = conn.execute(sql)
        cols = [d[0] for d in cur.description]
        rows = [dict(zip(cols, r)) for r in cur.fetchmany(50)]
        return {"count": len(rows), "rows": rows}
    except sqlite3.Error as e:
        return {"error": f"SQL error: {e}"}

SENT_EMAILS = []  # mock outbox -- real implementation would call smtplib / an email API

def send_email(to: str, subject: str, body: str) -> dict:
    SENT_EMAILS.append({"to": to, "subject": subject, "body": body})
    return {"status": "sent (mocked)", "to": to, "subject": subject}

def simulate_timeout_tool(**kwargs) -> dict:
    """A deliberately broken tool used in Section 7 to test error handling."""
    time.sleep(0.05)
    raise TimeoutError("Upstream service did not respond in time.")


Tool schemas + registry + RBAC

Schemas are what the model sees. `TOOL_PERMISSIONS` gates both (a) which
schemas get sent to the model per role, and (b) whether execution is
allowed even if a call slips through.

In [5]:
TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "search_internal_docs",
            "description": "Search internal company policy documents. Use for policy/process questions.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "query_database",
            "description": "Run a read-only SQL SELECT query against the 'orders' table "
                            "(columns: id, customer_email, status, total, created_at).",
            "parameters": {
                "type": "object",
                "properties": {"sql": {"type": "string"}},
                "required": ["sql"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email. Only call this after the user has confirmed the content.",
            "parameters": {
                "type": "object",
                "properties": {
                    "to": {"type": "string"},
                    "subject": {"type": "string"},
                    "body": {"type": "string"},
                },
                "required": ["to", "subject", "body"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "simulate_timeout_tool",
            "description": "Internal test tool that always times out (used to test error handling).",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

TOOL_REGISTRY = {
    "search_internal_docs": search_internal_docs,
    "query_database": query_database,
    "send_email": send_email,
    "simulate_timeout_tool": simulate_timeout_tool,
}

TOOL_PERMISSIONS = {
    "search_internal_docs": {"agent", "admin"},
    "send_email":           {"agent", "admin"},
    "query_database":       {"admin"},               # agents cannot query the DB directly
    "simulate_timeout_tool": {"agent", "admin"},
}

def get_tools_for_role(role: str) -> list[dict]:
    allowed = {n for n, roles in TOOL_PERMISSIONS.items() if role in roles}
    return [t for t in TOOL_SCHEMAS if t["function"]["name"] in allowed]


Executor with structured error handling

In [6]:
def execute_tool_call(name: str, arguments: dict, role: str) -> dict:
    if name not in TOOL_PERMISSIONS.get(name, set()) and role not in TOOL_PERMISSIONS.get(name, set()):
        return {"error": f"Role '{role}' is not permitted to use tool '{name}'.", "recoverable": False}

    fn = TOOL_REGISTRY.get(name)
    if fn is None:
        return {"error": f"Unknown tool '{name}'.", "recoverable": False}

    try:
        return {"result": fn(**arguments)}
    except TypeError as e:
        return {"error": f"Invalid arguments: {e}", "recoverable": True}
    except TimeoutError as e:
        return {"error": f"Tool timed out: {e}", "recoverable": True}
    except Exception as e:
        return {"error": f"Unexpected failure: {type(e).__name__}: {e}", "recoverable": False}


The LLM layer

Real OpenAI call if a key is set. Otherwise, a small scripted stub that
recognizes a few keyword patterns and proposes the same tool calls a real
model would -- this keeps every downstream cell (registry, RBAC, chaining,
error handling) genuinely exercised without requiring an API key.

In [7]:
def call_llm_openai(messages, tools):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
    return resp.choices[0].message

class StubMessage:
    """Mimics the subset of the OpenAI message object this notebook uses."""
    def __init__(self, content=None, tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls or []

class StubToolCall:
    def __init__(self, id, name, arguments):
        self.id = id
        self.function = type("F", (), {"name": name, "arguments": json.dumps(arguments)})

def _called_tool_names(messages) -> set:
    """Names of tools already called earlier in this conversation, read from
    the assistant messages we appended in agent_loop (each carries a
    'tool_calls' list of {id, name, arguments})."""
    names = set()
    for m in messages:
        if m.get("role") == "assistant":
            for tc in m.get("tool_calls", []):
                names.add(tc["name"])
    return names

def call_llm_stub(messages, tools):
    """Extremely simple scripted policy -- good enough to demo the pipeline,
    NOT a substitute for a real model."""
    last_user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    text = last_user.lower() if isinstance(last_user, str) else ""
    called = _called_tool_names(messages)

    if "email" in text and "order" in text:
        if "query_database" not in called:
            return StubMessage(tool_calls=[StubToolCall(
                "call_1", "query_database",
                {"sql": "SELECT * FROM orders WHERE customer_email='priya@example.com' ORDER BY created_at DESC LIMIT 1"},
            )])
        if "send_email" not in called:
            return StubMessage(tool_calls=[StubToolCall(
                "call_2", "send_email",
                {"to": "priya@example.com", "subject": "Order status update",
                 "body": "Your most recent order is currently delivered."},
            )])
        return StubMessage(content="Done -- I looked up your latest order and emailed you a status update.")

    if "refund" in text or "policy" in text:
        if "search_internal_docs" not in called:
            return StubMessage(tool_calls=[StubToolCall(
                "call_1", "search_internal_docs", {"query": text},
            )])
        return StubMessage(content="Based on our policy docs, damaged items can be refunded within 30 days.")

    if "timeout" in text or "broken tool" in text:
        return StubMessage(tool_calls=[StubToolCall("call_1", "simulate_timeout_tool", {})])

    if "database" in text and "raw sql" in text:
        return StubMessage(tool_calls=[StubToolCall(
            "call_1", "query_database", {"sql": "DROP TABLE orders"},  # deliberately invalid -- tests the guard
        )])

    return StubMessage(content="I'm not sure how to help with that in this demo -- try asking about refunds, "
                                 "or ask me to email a customer their order status.")

def call_llm(messages, tools):
    return call_llm_openai(messages, tools) if OPENAI_API_KEY else call_llm_stub(messages, tools)


Multi-turn agent loop

In [8]:
def agent_loop(user_message: str, role: str, max_turns: int = 5, verbose: bool = True) -> str:
    tools = get_tools_for_role(role)
    messages = [{"role": "user", "content": user_message}]

    for turn in range(1, max_turns + 1):
        msg = call_llm(messages, tools)

        if not msg.tool_calls:
            if verbose:
                print(f"[turn {turn}] final answer")
            return msg.content

        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": tc.id, "name": tc.function.name, "arguments": tc.function.arguments} for tc in msg.tool_calls
        ]})

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"[turn {turn}] tool call: {tc.function.name}({args})")
            result = execute_tool_call(tc.function.name, args, role)
            if verbose:
                print(f"[turn {turn}] tool result: {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    return "Max turns reached without a final answer."


Scenario: tool chaining (lookup -> email)

Two-step chain: query the DB for the customer's latest order, then email
them a status update using that result.

In [9]:
answer = agent_loop(
    "Please email priya@example.com an update about her most recent order.",
    role="admin",  # admin has query_database access; agent does not (see Section 9)
)
print("\nFINAL:", answer)
print("\nMocked outbox:", SENT_EMAILS)


[turn 1] tool call: query_database({'sql': "SELECT * FROM orders WHERE customer_email='priya@example.com' ORDER BY created_at DESC LIMIT 1"})
[turn 1] tool result: {'result': {'count': 1, 'rows': [{'id': 'ord_1002', 'customer_email': 'priya@example.com', 'status': 'delivered', 'total': 120.0, 'created_at': '2026-07-28'}]}}
[turn 2] tool call: send_email({'to': 'priya@example.com', 'subject': 'Order status update', 'body': 'Your most recent order is currently delivered.'})
[turn 2] tool result: {'result': {'status': 'sent (mocked)', 'to': 'priya@example.com', 'subject': 'Order status update'}}
[turn 3] final answer

FINAL: Done -- I looked up your latest order and emailed you a status update.

Mocked outbox: [{'to': 'priya@example.com', 'subject': 'Order status update', 'body': 'Your most recent order is currently delivered.'}]


Role-based access control in action

Same request, but as `agent` (no `query_database` permission). The tool
schema for `query_database` isn't even sent to the model for this role, and
the executor double-checks permissions independently.

In [10]:
print("Tools visible to 'agent' role:", [t["function"]["name"] for t in get_tools_for_role("agent")])
print("Tools visible to 'admin' role:", [t["function"]["name"] for t in get_tools_for_role("admin")])

# Direct executor-level check (defense in depth) -- simulate a call slipping through
blocked = execute_tool_call("query_database", {"sql": "SELECT * FROM orders"}, role="agent")
print("\nAgent attempting query_database directly:", blocked)


Tools visible to 'agent' role: ['search_internal_docs', 'send_email', 'simulate_timeout_tool']
Tools visible to 'admin' role: ['search_internal_docs', 'query_database', 'send_email', 'simulate_timeout_tool']

Agent attempting query_database directly: {'error': "Role 'agent' is not permitted to use tool 'query_database'.", 'recoverable': False}


Deliberate failure cases

Three scenarios that exercise `execute_tool_call`'s structured error
handling instead of crashing the notebook.

In [11]:
# 1. Unauthorized tool call (role check)
print("Unauthorized call:", execute_tool_call("query_database", {"sql": "SELECT * FROM orders"}, role="agent"))

# 2. Bad arguments (TypeError path)
print("\nBad arguments:", execute_tool_call("send_email", {"to": "x@example.com"}, role="admin"))  # missing subject/body

# 3. Simulated timeout
print("\nSimulated timeout:", execute_tool_call("simulate_timeout_tool", {}, role="admin"))

# 4. Guardrail on non-SELECT SQL
print("\nDestructive SQL blocked:", execute_tool_call("query_database", {"sql": "DROP TABLE orders"}, role="admin"))


Unauthorized call: {'error': "Role 'agent' is not permitted to use tool 'query_database'.", 'recoverable': False}

Bad arguments: {'error': "Invalid arguments: send_email() missing 2 required positional arguments: 'subject' and 'body'", 'recoverable': True}

Simulated timeout: {'error': 'Tool timed out: Upstream service did not respond in time.', 'recoverable': True}

Destructive SQL blocked: {'result': {'error': 'Only SELECT statements are allowed.'}}
